# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below:

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

The dataset functions implemented in Chapter 12 can be easily imported from a set of helper functions we're making available for your convenience. You can download it from the following link:

`https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py`

In Google Colab, you can run the following command to download the file:

In [ ]:
!wget https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py

Once the file is downloaded, you only need to import the required functions:

In [1]:
from object_detection_functions import *

## 12.8 Lab 5A: Overview

In this first part of Lab 5, you will prepare the Oxford-IIIT Pet dataset for object detection.

You will parse the XML annotations, build COCO-style metadata, create a Hugging Face Dataset, attach Torchvision v2 transformations, and create the DataLoaders that will be consumed by the detector in Lab 5B.

### 12.8.1 Oxford-IIIT Pet Dataset

You'll build a dataset using the images and annotations from the Oxford-IIIT Pet dataset:

> "_We have created a 37 category pet dataset with roughly 200 images for each class. The images have a large variations in scale, pose and lighting. All images have an associated ground truth annotation of breed, head ROI, and pixel level trimap segmentation._"

First, we download images and annotations for the dataset and extract them locally:


In [2]:
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
# Alternative mirror
# !wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/annotations.tar.gz
# !wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/images.tar.gz

--2026-11-18 18:44:48--  https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)|129.67.95.98|:443... connected.
HTTP request sent, awaiting response... 308 Permanent Redirect
Location: https://thor.robots.ox.ac.uk/pets/annotations.tar.gz [following]
--2026-11-18 18:44:48--  https://thor.robots.ox.ac.uk/pets/annotations.tar.gz
Reusing existing connection to thor.robots.ox.ac.uk:443.
HTTP request sent, awaiting response... 200 OK
Length: 19173078 (18M) [application/octet-stream]
Saving to: ‘annotations.tar.gz’

annotations.tar.gz  100%[===================>]  18.28M  20.6MB/s    in 0.9s    

2026-11-18 18:44:49 (20.6 MB/s) - ‘annotations.tar.gz’ saved [19173078/19173078]

--2026-11-18 18:44:49--  https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.u

In [3]:
import tarfile
# Open and extract the archive automatically detecting compression (mode='r:*')
with tarfile.open('annotations.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Annotations extracted successfully!")
with tarfile.open('images.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Images extracted successfully!")

/tmp/ipykernel_565/12807389.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Annotations extracted successfully!


/tmp/ipykernel_565/12807389.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Images extracted successfully!


### 12.8.2 Annotations

The annotations follow the Pascal VOC challenge format, and are stored as individual XML files, one for each annotated image, inside the oxford-iiit-pet/annotations/xmls subfolder. Use the `xml_to_csv()` helper function to convert all these files into a Pandas dataframe and inspect its contents.

In [4]:
# write your code here
xml_df = xml_to_csv("annotations/xmls")
xml_df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,american_bulldog_14.jpg,500,376,dog,92,22,369,343
1,american_pit_bull_terrier_153.jpg,300,230,dog,15,10,108,120
2,Egyptian_Mau_138.jpg,448,336,cat,283,15,411,142
3,British_Shorthair_186.jpg,500,375,cat,148,33,349,245
4,Persian_193.jpg,500,436,cat,311,109,467,265


The annotations contain the box coordinates in the Pascal VOC system (`[xmin, ymin, xmax, ymax]`), but they only have two main classes, cats and dogs, instead of the expected 37 classes found in the description. As it turns out, there are more files in the annotations folder, namely, `list.txt`, `trainval.txt`, and `test.txt`.

If you're in Google Colab, the command below will list the files inside the annotations folder:

In [6]:
# if you chose a different root folder, change it accordingly
!ls -l ./annotations

total 1076
-rw-r--r-- 1 501 staff 172300 Jun 29  2012 list.txt
-rw-r--r-- 1 501 staff   1476 Jun 30  2012 README
-rw-r--r-- 1 501 staff  84319 Jun 29  2012 test.txt
-rw-r--r-- 1 501 staff  87776 Jun 29  2012 trainval.txt
drwxr-xr-x 2 501 staff 610304 Jun 18  2012 trimaps
drwxr-xr-x 2 501 staff 135168 Jun 29  2012 xmls


Let's take a look at the list.txt file.

If you're in Google Colab, the command below will show you the first few lines of the list.txt file:

In [7]:
!head ./annotations/list.txt

#Image CLASS-ID SPECIES BREED ID
#ID: 1:37 Class ids
#SPECIES: 1:Cat 2:Dog
#BREED ID: 1-25:Cat 1:12:Dog
#All images with 1st letter as captial are cat images
#images with small first letter are dog images
Abyssinian_100 1 1 1
Abyssinian_101 1 1 1
Abyssinian_102 1 1 1
Abyssinian_103 1 1 1


It contains a list of all images in the dataset, organized in four columns separated by spaces: Image, CLASS-ID, SPECIES, BREED ID. As it turns out, the "class" from the XML file is actually the species. We're interested in the true class ids, from 1 to 37, as stated in the description.

Now, let's take a look at the file corresponding to the data you loaded, the trainval split. If you're in Google Colab, the command below will show you the first few lines of the trainval.txt file:

In [8]:
!head ./annotations/trainval.txt

Abyssinian_100 1 1 1
Abyssinian_101 1 1 1
Abyssinian_102 1 1 1
Abyssinian_103 1 1 1
Abyssinian_104 1 1 1
Abyssinian_105 1 1 1
Abyssinian_106 1 1 1
Abyssinian_107 1 1 1
Abyssinian_108 1 1 1
Abyssinian_109 1 1 1


It clearly follows the same structure as the previous file, but it does not contain any headers, and it lists only the images that belong to the original train and validation split.

We can load it in Pandas for easier visualization (just run the code below as is to visualize the dataframe with the information from the trainval.txt file):

In [9]:
import pandas as pd

trainval_df = pd.read_csv('annotations/trainval.txt', sep=' ', header=None, names=['filename', 'class_id', 'species', 'breed_id'])
trainval_df

,filename,class_id,species,breed_id
0,Abyssinian_100,1,1,1
1,Abyssinian_101,1,1,1
2,Abyssinian_102,1,1,1
3,Abyssinian_103,1,1,1
4,Abyssinian_104,1,1,1
...,...,...,...,...
3675,yorkshire_terrier_187,37,2,25
3676,yorkshire_terrier_188,37,2,25
3677,yorkshire_terrier_189,37,2,25
3678,yorkshire_terrier_18,37,2,25


Each filename has its own corresponding class index (class_id), but the label itself, as the descriptive name corresponding to the category, is only available as part of the filename itself.

We can easily extract it, though. Just run the code below as is to create a new column (category) in the dataframe:


In [10]:
trainval_df['category'] = trainval_df['filename'].apply(lambda v: ' '.join([w.capitalize()
                                                                            for w in v.split('_')[:-1]]))

Moreover, there are 3,680 rows, one for each image, but there are 3,687 annotations retrieved from the XML files. Why? It is important to highlight that:
- some images may have more than one annotation/box - you saw that already in the Penn-Fudan dataset
- some images probably have no annotations/boxes (you'll see that soon)

We'll use the same custom dataset class ObjDetectionDataset once again, since it is prepared to take a CSV file or Pandas dataframe containing the annotations (filename, labels, xmin, ymin, xmax, and ymax columns), but keep in mind that only the filenames in the file/dataframe are going to be considered by it.

Therefore, we need to build an annotations file/dataframe that includes filenames that have no annotations as well. It is better to keep images without annotations as negative cases, so we merge both dataframes and make sure that:
- every filename is kept, so there are still 3,680 unique filenames after merging
- the resulting dataframe has, at least, the following columns: filename, label, category, xmin, ymin, xmax, and ymax

Run the code below as is to build the corresponding dataframe of annotations:

In [11]:
trainval_df['filename'] = trainval_df['filename'].apply(lambda v: f'{v}.jpg')
annotations_df = trainval_df.merge(xml_df, how='left', on='filename')

colnames = ['filename', 'label', 'category', 'width', 'height', 'xmin', 'ymin', 'xmax', 'ymax']
annotations_df = annotations_df.rename(columns={'class_id': 'label'})[colnames]
annotations_df

,filename,label,category,width,height,xmin,ymin,xmax,ymax
0,Abyssinian_100.jpg,1,Abyssinian,394.0,500.0,151.0,71.0,335.0,267.0
1,Abyssinian_101.jpg,1,Abyssinian,450.0,313.0,54.0,36.0,319.0,235.0
2,Abyssinian_102.jpg,1,Abyssinian,500.0,465.0,23.0,27.0,325.0,320.0
3,Abyssinian_103.jpg,1,Abyssinian,500.0,351.0,241.0,68.0,362.0,196.0
4,Abyssinian_104.jpg,1,Abyssinian,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3676,yorkshire_terrier_187.jpg,37,Yorkshire Terrier,500.0,334.0,187.0,49.0,317.0,181.0
3677,yorkshire_terrier_188.jpg,37,Yorkshire Terrier,500.0,334.0,94.0,36.0,378.0,263.0
3678,yorkshire_terrier_189.jpg,37,Yorkshire Terrier,500.0,334.0,274.0,84.0,402.0,207.0
3679,yorkshire_terrier_18.jpg,37,Yorkshire Terrier,500.0,375.0,52.0,23.0,289.0,183.0


Besides, we'll use the resulting dataframe to build a id2label dictionary to map class id into the corresponding category. Run the code below as is to build the dictionary:

In [12]:
id2label = dict(annotations_df[['label', 'category']].drop_duplicates().values)
id2label

{1: 'Abyssinian',
 2: 'American Bulldog',
 3: 'American Pit Bull Terrier',
 4: 'Basset Hound',
 5: 'Beagle',
 6: 'Bengal',
 7: 'Birman',
 8: 'Bombay',
 9: 'Boxer',
 10: 'British Shorthair',
 11: 'Chihuahua',
 12: 'Egyptian Mau',
 13: 'English Cocker Spaniel',
 14: 'English Setter',
 15: 'German Shorthaired',
 16: 'Great Pyrenees',
 17: 'Havanese',
 18: 'Japanese Chin',
 19: 'Keeshond',
 20: 'Leonberger',
 21: 'Maine Coon',
 22: 'Miniature Pinscher',
 23: 'Newfoundland',
 24: 'Persian',
 25: 'Pomeranian',
 26: 'Pug',
 27: 'Ragdoll',
 28: 'Russian Blue',
 29: 'Saint Bernard',
 30: 'Samoyed',
 31: 'Scottish Terrier',
 32: 'Shiba Inu',
 33: 'Siamese',
 34: 'Sphynx',
 35: 'Staffordshire Bull Terrier',
 36: 'Wheaten Terrier',
 37: 'Yorkshire Terrier'}

Let's run some assert commands to ensure everything is as expected. Run the code below as is. It shouldn't raise any errors nor produce any output. If an error is raised, you should double-check the code for loading the dataset and its annotations.

In [13]:
assert len(annotations_df['filename'].unique()) == 3680
assert len(id2label.values()) == 37
assert len(annotations_df) == 3681

Shouldn'it be 3,687? Perhaps even more, since it should also include images without any annotations? It actually should, but some of the annotated images were excluded from the trainval.txt list of files for some unknown reason. In case you're curious, these are the images.

Run the code below as is to visualize the extra annotations:

In [14]:
extra_annotations = set(xml_df['filename'].unique()).difference(set(annotations_df['filename'].unique()))
extra_annotations

{'Bombay_11.jpg',
 'Bombay_189.jpg',
 'Bombay_190.jpg',
 'Bombay_192.jpg',
 'Egyptian_Mau_129.jpg',
 'Egyptian_Mau_183.jpg',
 'Siamese_203.jpg',
 'english_cocker_spaniel_162.jpg',
 'english_cocker_spaniel_163.jpg',
 'english_cocker_spaniel_164.jpg',
 'english_cocker_spaniel_179.jpg',
 'newfoundland_152.jpg',
 'newfoundland_153.jpg',
 'newfoundland_154.jpg',
 'newfoundland_155.jpg'}

The whole point of this apparent detour from our main job here - fine-tuning an object detection model - is to illustrate the fact that every dataset has its issues, and you should always take your time to investigate how it's organized, if there are quality issues, and ensure it's in the right shape to be loaded into an instance of your dataset class.

So, let's get rid of any annotation that has something missing:

In [15]:
annotations_df = annotations_df.dropna().copy()

### 12.8.3 Building the Dataset

The XML files use opposite corners. We’ll compute width and height so each box can be stored as `[x, y, width, height]`. We’ll also convert the original labels from 1..37 to the zero-based 0..36 convention used in the Hugging Face dataset.

In [16]:
# write your code here
annotations_df["box_width"] = (annotations_df["xmax"] - annotations_df["xmin"])
annotations_df["box_height"] = (annotations_df["ymax"] - annotations_df["ymin"])

# make classes zero-based for our HF-style dataset
# write your code here
annotations_df["label"] = annotations_df["label"] - 1

Next, we’ll create one metadata record per image, keeping bounding boxes and categories as lists so the representation also works for images with multiple objects.

In [17]:
import json

records = []
for filename, group in annotations_df.groupby("filename", sort=False):
    objects = {
        "bbox": [[float(row.xmin), float(row.ymin), float(row.box_width), float(row.box_height)]
                for row in group.itertuples()],
        "category": [int(label) for label in group["label"]],
    }
    record = {
        "file_name": filename,
        "width": int(group.iloc[0]["width"]),
        "height": int(group.iloc[0]["height"]),
        "objects": objects,
        "label": int(group.iloc[0]["label"])
    }
    records.append(record)

records[0]

{'file_name': 'Abyssinian_100.jpg',
 'width': 394,
 'height': 500,
 'objects': {'bbox': [[151.0, 71.0, 184.0, 196.0]], 'category': [0]},
 'label': 0}

Then, we save the metadata to a file named metadata.jsonl, and use it as an argument of our `build_object_detection_dataset()` helper function (developed in Chapter 12).

In [18]:
from pathlib import Path

metadata_file = Path("metadata.jsonl")
with metadata_file.open("w", encoding="utf-8") as f:
    f.writelines([json.dumps(record) + '\n' for record in records])

In [19]:
image_folder = './images'
# write your code here
pets_detection = build_object_detection_dataset(image_folder, metadata_file=metadata_file)

In order to split the dataset in a stratified fashion, so every class is represented in every split, we must cast the image-level breed label to `ClassLabel` first:

In [20]:
from datasets import ClassLabel
pets_detection = pets_detection.cast_column("label", ClassLabel(num_classes=37))

Casting the dataset:   0%|          | 0/3671 [00:00<?, ? examples/s]

Let’s check one example from our dataset now:

In [21]:
pets_detection[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=394x500>,
 'image_id': 0,
 'width': 394,
 'height': 500,
 'objects': {'area': [36064.0],
  'bbox': [[151.0, 71.0, 184.0, 196.0]],
  'category': [0],
  'id': [0],
  'iscrowd': [0]},
 'label': 0}

### 12.8.4 Train-Validation Splits

The original list of files does not give any indication regarding the split between training and validation sets, so you'll have to do it yourself.

Our suggestion is to shuffle the filenames, and take a large part of them (e.g. 80%) as training set, and the remaining files as validation set. Do not forget to stratify it by label (using the stratify_by_column argument):

In [22]:
# write your code here
datasets = pets_detection.shuffle(seed=42).train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
datasets["validation"] = datasets.pop("test")
datasets

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects', 'label'],
        num_rows: 2936
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'objects', 'label'],
        num_rows: 735
    })
})

### 12.8.5 Data Augmentation

It is time to write your own training and validation transformation functions. First, define the composed transformations for each split:

- if it is the validation set, it should stick to the basics
    - sanitize bounding boxes, just in case
    - convert, if needed, PIL images to tensors
    - normalize the values
- if it is in the training set, it may perform data augmentation as well:
    - choose one or more data augmenting transformations
    - sanitize bounding boxes, just in case
    - convert, if needed, PIL images to tensors

Then, use them as arguments to  our own `build_detection_transform()` developed in Chapter 12, which builds the output dictionary expected by an object detection model.

In [23]:
from torchvision.transforms import v2

# write your code here
train_transforms = v2.Compose([v2.RandomHorizontalFlip(p=0.5), v2.SanitizeBoundingBoxes(), v2.ToDtype(torch.float32, scale=True)])
val_transforms = v2.Compose([v2.SanitizeBoundingBoxes(), v2.ToDtype(torch.float32, scale=True)])

# write your code here
train_fn = build_detection_transform(train_transforms)
val_fn = build_detection_transform(val_transforms)

### 12.8.6 Datasets and DataLoaders

The transformation functions can be lazily attached to each corresponding split using the datasets `with_transform()` method:

In [25]:
# write your code here
datasets['train'] = datasets['train'].with_transform(train_fn)
datasets['validation'] = datasets['validation'].with_transform(val_fn)
len(datasets['train']), len(datasets['validation'])

(2936, 735)

Next, create two data loaders, one for each dataset. You should shuffle the training set, but not the validation one. Also, keep batch size small (e.g. four) to avoid out-of-memory issues in the GPU.

In [26]:
from torch.utils.data import DataLoader

dataloaders = {}
# write your code here
dataloaders['train'] = DataLoader(datasets['train'], batch_size=4, shuffle=True)
dataloaders['validation'] = DataLoader(datasets['validation'], batch_size=4, shuffle=False)

Try fetching a mini-batch from your training set. Just run the code below as is:

In [27]:
next(iter(dataloaders['train']))

RuntimeError: stack expects each tensor to be equal size, but got [3, 386, 500] at entry 0 and [3, 333, 500] at entry 1

Did you get an error? No? Consider yourself lucky! At some point, it will raise an error, whenever it tries to stack images with different sizes.

The collate function is the function used by the data loader to patch together multiple data points into a mini-batch. If your dataset is nothing but tensors, that's trivial: it only has to stack them up. Stacking them up, though, assumes every data point has exactly the same shape for its features.

In object detection models, though, this is not guaranteed to be the case: not only images may have different sizes, but also one image may have no boxes, another one may have three boxes, and yet another one may have only one. Those cannot be stacked together.
The solution, fortunately, is pretty easy, and it looks like this:

In [28]:
def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = [item["target"] for item in batch]

    return images, targets

Throw the function above as the `collate_fn` argument of your data loaders, and try again:

In [29]:
dataloaders = {}
# write your code here
dataloaders['train'] = DataLoader(datasets['train'], batch_size=4, shuffle=True, collate_fn=collate_fn)
dataloaders['validation'] = DataLoader(datasets['validation'], batch_size=4, shuffle=False, collate_fn=collate_fn)

In [30]:
next(iter(dataloaders['train']))

([Image([[[0.4941, 0.5176, 0.3725,  ..., 0.1294, 0.1373, 0.1098],
          [0.5137, 0.5294, 0.3686,  ..., 0.1333, 0.1451, 0.1216],
          [0.5216, 0.5216, 0.3529,  ..., 0.1333, 0.1451, 0.1294],
          ...,
          [0.3020, 0.3059, 0.3059,  ..., 0.3020, 0.3176, 0.2863],
          [0.2980, 0.3020, 0.3098,  ..., 0.2549, 0.2863, 0.2706],
          [0.3098, 0.3098, 0.3176,  ..., 0.2353, 0.2235, 0.2353]],
  
         [[0.2745, 0.3176, 0.2196,  ..., 0.1176, 0.1137, 0.0863],
          [0.2941, 0.3294, 0.2157,  ..., 0.1216, 0.1216, 0.0980],
          [0.3098, 0.3333, 0.2039,  ..., 0.1216, 0.1216, 0.1059],
          ...,
          [0.0784, 0.0784, 0.0784,  ..., 0.2392, 0.2627, 0.2235],
          [0.0745, 0.0745, 0.0706,  ..., 0.2000, 0.2314, 0.2157],
          [0.0824, 0.0824, 0.0706,  ..., 0.2078, 0.1961, 0.2078]],
  
         [[0.1843, 0.2353, 0.1529,  ..., 0.0588, 0.0510, 0.0235],
          [0.2039, 0.2471, 0.1490,  ..., 0.0627, 0.0588, 0.0353],
          [0.2157, 0.2471, 0.1294,  ..